# Step selection, Run 1: the steps 1–3 vote and a hidden-state probe, on the GSM8K test set

This is the last test of step-level selection. It is evaluation only, it **uses the 1,319-question test set**, and every
criterion is frozen in `results/step_select/run1/PLAN.md`. The notebook refuses to start unless that file matches
`PLAN.sha256`.

- **The run:** greedy, plus 7 MC-dropout reruns (p = 0.2) at each of steps 1–3, each finished normally. Each candidate's
  hidden state and latent are saved.
- **A:** does voting over greedy and the first 3 reruns at steps 1–3 beat greedy? CONFIRM needs a net of at least +8 and a
  question-level p < 0.05.
- **B:** can a probe on hidden states pick right reruns? It is cross-fitted over 5 question folds.
  - GO needs a within-set AUC of at least 0.65, a net of at least +27 and p < 0.05.
  - REPORT means the AUC passes but the gain doesn't.
  - KILL means the AUC is below 0.65.
- **Preflight:** on the first 50 questions, greedy must match the held-out study's test answers on 50/50, and the full run
  must project to 15 minutes or less.
- **Cap:** 20 minutes of evaluation in total, and at most one resumed session.
- **Afterwards,** the test set is spent for step selection.

**Before running:**
1. Runtime → Change runtime type → **G4**.
2. Set `RUN_TEST = True` below. This is the explicit opt-in for using the test set.
3. Runtime → **Run all**.

In [ ]:
RUN_TEST = False          # the explicit test-set opt-in (PLAN.md); nothing runs while False
REPO_REF = "main"         # the branch or commit holding scripts/step_select/run1.py and the frozen plan

In [ ]:
from pathlib import Path
import hashlib, json, os, runpy, shutil, subprocess, sys
from google.colab import drive

if not RUN_TEST:
    raise SystemExit("RUN_TEST is False: set it to True to run on the test set (results/step_select/run1/PLAN.md).")
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/dynamic-state-anchoring")
BACKUP_DIR = DRIVE / "step-select" / "run1"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode:
    raise SystemExit("No GPU: Runtime > Change runtime type > GPU, then Runtime > Run all.")
print("GPU:", gpu.stdout.strip())

PINNED = BACKUP_DIR / "commit.txt"  # a started run keeps the code it started with
ref = PINNED.read_text().strip() if PINNED.exists() else REPO_REF
REPO_DIR = Path("/content/dynamic-state-anchoring")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/sttruji/dynamic-state-anchoring.git", str(REPO_DIR)],
                   check=True)
subprocess.run(["git", "fetch", "origin", ref], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
plan = Path("results/step_select/run1/PLAN.md")
pinned = Path("results/step_select/run1/PLAN.sha256").read_text().split()[0]
if hashlib.sha256(plan.read_bytes()).hexdigest() != pinned:
    raise SystemExit("PLAN.md does not match PLAN.sha256: the frozen plan was changed; stop.")
BACKUP_DIR.mkdir(parents=True, exist_ok=True)
PINNED.write_text(COMMIT)
sessions = BACKUP_DIR / "sessions.txt"
count = int(sessions.read_text()) + 1 if sessions.exists() else 1
if count > 2:
    raise SystemExit("Run 1 allows one session and one resume (PLAN.md); this would be a third.")
sessions.write_text(str(count))
print("Code:", COMMIT, f"(from {ref}); plan sha256 matches; session {count} of at most 2")

In [ ]:
VENV = Path("/content/dsa-venv")
subprocess.run([sys.executable, "scripts/setup_colab.py", "--venv-dir", str(VENV),
                "--requirements", "requirements-colab.txt"], check=True)
PYTHON = str(VENV / "bin/python")
os.environ["DSA_DATA_DIR"] = str(DRIVE / "data")
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
run_logged = runpy.run_path("scripts/colab_process.py")["run_logged"]

def run(*args):
    run_logged([PYTHON, *map(str, args)], cwd=REPO_DIR, log_dir=BACKUP_DIR / "logs")

run("scripts/setup_data.py", "--eval-only")  # the evaluation sets, the test set included
CHECKPOINT = DRIVE / "checkpoints/simcot-gpt2"
run("scripts/latent_gate/fetch_simcot.py", "--output", CHECKPOINT)
REFERENCE = "results/latent_gate/heldout_scale/simcot-gpt2/gsm8k_test.jsonl"
CAP = 20 * 60  # seconds of evaluation in total (PLAN.md)

def spent():  # evaluation seconds used so far, the preflight included, across both possible run folders
    paths = [BACKUP_DIR / name / "status.json" for name in ("run", "run-k3")]
    return sum(json.loads(p.read_text())["seconds"] for p in paths if p.exists())

def evaluate(folder, reruns, limit=None):
    run("scripts/step_select/run1.py", "--checkpoint", CHECKPOINT, "--dataset", "gsm8k_test", "--output", folder,
        "--reference", REFERENCE, "--reruns", reruns, "--device", "cuda",
        "--max-minutes", max(0.0, CAP - spent()) / 60, *([] if limit is None else ["--limit", limit]))
    return json.loads((folder / "status.json").read_text())

In [ ]:
# Preflight: the first 50 questions, kept as the start of the full run when it passes.
RUN = BACKUP_DIR / "run"
decision = BACKUP_DIR / "preflight.json"
if decision.exists():
    choice = json.loads(decision.read_text())
else:
    status = evaluate(RUN, 7, limit=50)
    if status["identity"] != 50:
        raise SystemExit(f"Preflight identity failed: greedy matches the reference on {status['identity']}/50. Stop.")
    projected = status["seconds"] * 1319 / 50
    choice = {"projected_seconds": projected, "reruns": 7 if projected <= 15 * 60 else 3,
              "folder": "run" if projected <= 15 * 60 else "run-k3"}
    decision.write_text(json.dumps(choice, indent=1))
print(f"Preflight: identity 50/50, full run projected at {choice['projected_seconds'] / 60:.1f} minutes -> "
      f"{choice['reruns']} reruns per step" + ("" if choice["reruns"] == 7 else
      "; B does not run, and Stage 0b becomes Run 2 (PLAN.md)"))

In [ ]:
FINAL = BACKUP_DIR / choice["folder"]
status = evaluate(FINAL, choice["reruns"])
print(json.dumps(status, indent=1))

In [ ]:
report = subprocess.run([PYTHON, "scripts/step_select/run1_report.py", str(FINAL)], cwd=REPO_DIR,
                        capture_output=True, text=True)
print(report.stdout, report.stderr)
(BACKUP_DIR / "report.txt").write_text(report.stdout)

In [ ]:
# The records, status, report and features, for the repo's results folder.
from google.colab import files

bundle = Path("/content/step-select-run1")
shutil.rmtree(bundle, ignore_errors=True)
shutil.copytree(FINAL, bundle / FINAL.name)
for name in ("report.txt", "preflight.json", "commit.txt", "sessions.txt"):
    shutil.copy2(BACKUP_DIR / name, bundle / name)
files.download(shutil.make_archive(str(bundle), "zip", bundle))
print(f"Downloading {bundle}.zip (about 100 MB with the features). If it does not start, use the Files panel.")

**Reading the result**
- **Preflight** must show identity 50/50. If the run was downgraded to 3 reruns, B is skipped and Stage 0b would be next.
- **A** prints CONFIRM or KILL. NO VERDICT means the run was incomplete or greedy failed to match somewhere.
- **B** prints the within-set AUC, the candidate-0 detector's AUC, the selector's net and p, the learning curve, and GO,
  REPORT or KILL.
- After this run, the test set is spent for step selection, whatever the outcome.

**Sharing:** the last cell downloads a zip. Point Claude at it.